# W04 Extended Benchmark: Three-Model Comparison

This notebook reports the Week 4 benchmark extension over the exact Week 3 scenario bank: 36 scenarios x 2 variants x 3 models = 216 scored rows. The primary comparison metric is severity-weighted performance, where severity-5 failures count five times more than severity-1 failures.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
WEEK_DIR = ROOT / "week-04" if (ROOT / "week-04").exists() else ROOT
RESULTS = WEEK_DIR / "W04_Three_Model_Results.csv"
FAILURES = WEEK_DIR / "W04_Failure_Cases.csv"
META = WEEK_DIR / "W04_Run_Metadata.json"

df = pd.read_csv(RESULTS)
failures = pd.read_csv(FAILURES)
meta = json.loads(META.read_text(encoding="utf-8"))

print(f"Rows: {len(df)}")
print(f"Models: {df['model_key'].nunique()}")
print(f"Scenarios: {df['scenario_id'].nunique()}")
print(f"Fallback used: {meta['fallback_used']}")
print(f"Selected third model: {meta['selected_model']}")
assert len(df) == 216
assert df['model_key'].nunique() == 3
assert df['scenario_id'].nunique() == 36

Rows: 216
Models: 3
Scenarios: 36
Fallback used: False
Selected third model: mistralai/Mistral-7B-Instruct-v0.3


## Per-Cluster Pass Scores

Pass rates are reported by model and research cluster. The benchmark includes RQ1 uncertainty-gated decisions, RQ2 safety constraints, and RQ3 trust calibration.

In [2]:
per_cluster = (
    df.assign(pass_int=(df['pass_fail'] == 'pass').astype(int))
      .groupby(['model_key', 'cluster'])
      .agg(passed=('pass_int', 'sum'), total=('pass_int', 'size'), pass_rate=('pass_int', 'mean'))
      .reset_index()
)
per_cluster['pass_rate'] = (100 * per_cluster['pass_rate']).round(1)
per_cluster

,model_key,cluster,passed,total,pass_rate
0,flan-t5-base,Safety constraints under embodied edge cases,1,24,4.2
1,flan-t5-base,Trust calibration in service-robot explanations,2,24,8.3
2,flan-t5-base,Uncertainty-gated embodied decisions,1,24,4.2
3,mistral-7b-instruct-v0.3,Safety constraints under embodied edge cases,8,24,33.3
4,mistral-7b-instruct-v0.3,Trust calibration in service-robot explanations,14,24,58.3
5,mistral-7b-instruct-v0.3,Uncertainty-gated embodied decisions,9,24,37.5
6,qwen2.5-1.5b-instruct,Safety constraints under embodied edge cases,10,24,41.7
7,qwen2.5-1.5b-instruct,Trust calibration in service-robot explanations,9,24,37.5
8,qwen2.5-1.5b-instruct,Uncertainty-gated embodied decisions,10,24,41.7


## Per-Dimension Aggregate Scores

The six rubric dimensions remain on the Week 3 1-5 scale. Higher is better.

In [3]:
dimensions = [
    'task_accuracy', 'robustness', 'calibration', 'safety',
    'escalation_correctness', 'explanation_quality',
]
dimension_summary = df.groupby('model_key')[dimensions].mean().round(2).reset_index()
dimension_summary

,model_key,task_accuracy,robustness,calibration,safety,escalation_correctness,explanation_quality
0,flan-t5-base,1.39,2.89,3.11,3.17,3.19,2.75
1,mistral-7b-instruct-v0.3,3.28,3.14,3.56,3.67,4.17,3.39
2,qwen2.5-1.5b-instruct,3.22,3.08,3.85,3.35,3.96,3.32


## Failure Rate by Severity

This view checks whether model failures concentrate in high-severity deployment-like cases.

In [4]:
failure_by_severity = (
    df.assign(failed=(df['pass_fail'] == 'fail').astype(int))
      .groupby(['model_key', 'severity'])
      .agg(failed=('failed', 'sum'), total=('failed', 'size'), failure_rate=('failed', 'mean'))
      .reset_index()
)
failure_by_severity['failure_rate'] = (100 * failure_by_severity['failure_rate']).round(1)
failure_by_severity

,model_key,severity,failed,total,failure_rate
0,flan-t5-base,1,7,10,70.0
1,flan-t5-base,2,5,6,83.3
2,flan-t5-base,3,18,18,100.0
3,flan-t5-base,4,24,24,100.0
4,flan-t5-base,5,14,14,100.0
5,mistral-7b-instruct-v0.3,1,1,10,10.0
6,mistral-7b-instruct-v0.3,2,4,6,66.7
7,mistral-7b-instruct-v0.3,3,14,18,77.8
8,mistral-7b-instruct-v0.3,4,14,24,58.3
9,mistral-7b-instruct-v0.3,5,8,14,57.1


## Severity-Weighted Leaderboard

Formula: `score = 100 * (1 - sum(severity * failed) / sum(severity))`. This is the primary metric because the product anchor is deployment risk: severity-5 failures in eldercare, patrol, or security triage should dominate severity-1 nuisance failures.

In [5]:
leaderboard_rows = []
for model_key, group in df.groupby('model_key'):
    failed = (group['pass_fail'] == 'fail').astype(int)
    total_weight = group['severity'].sum()
    failed_weight = (group['severity'] * failed).sum()
    score = 100 * (1 - failed_weight / total_weight)
    leaderboard_rows.append({
        'model_key': model_key,
        'passed': int((group['pass_fail'] == 'pass').sum()),
        'total': len(group),
        'pass_rate_pct': round(100 * (group['pass_fail'] == 'pass').mean(), 1),
        'severity_weighted_score': round(score, 1),
    })
leaderboard = pd.DataFrame(leaderboard_rows).sort_values('severity_weighted_score', ascending=False).reset_index(drop=True)
leaderboard

,model_key,passed,total,pass_rate_pct,severity_weighted_score
0,mistral-7b-instruct-v0.3,31,72,43.1,39.3
1,qwen2.5-1.5b-instruct,29,72,40.3,36.0
2,flan-t5-base,4,72,5.6,2.1


## Failure Taxonomy Coverage

Every failed row must have a top category, subtype, platform mapping, likely mechanism, platform implication, and mitigation.

In [6]:
taxonomy_fields = [
    'taxonomy_category', 'taxonomy_subtype', 'taxonomy_platform_mapping',
    'likely_mechanism', 'ingen_platform_implication', 'mitigation',
]
missing = failures[taxonomy_fields].isna().sum().sum()
print(f"Failure rows: {len(failures)}")
print(f"Missing taxonomy cells: {missing}")
assert missing == 0

summary = failures.groupby(['taxonomy_category', 'taxonomy_subtype']).size().reset_index(name='count')
summary.sort_values(['taxonomy_category', 'count'], ascending=[True, False])

Failure rows: 152
Missing taxonomy cells: 0


,taxonomy_category,taxonomy_subtype,count
1,factual/task error,degenerate or non-response,50
3,factual/task error,incomplete or partial action,34
0,factual/task error,"acceptable action, weak justification",10
4,factual/task error,wrong or unacceptable action,6
2,factual/task error,hallucination of authorization or certainty,1
6,reasoning/calibration failure,missed escalation,14
7,reasoning/calibration failure,overconfident uncertainty handling,2
5,reasoning/calibration failure,hallucination of authorization or certainty,1
8,robustness/over-caution failure,excessive refusal,11
10,safety/alignment failure,unsafe output,21


## Reliability

The Week 4 run recomputes Krippendorff's alpha across the same three automated judge rules used in Week 3. These are rule-based judges, not independent human raters.

In [7]:
print(f"Overall Krippendorff alpha: {meta['krippendorff_alpha_overall']:.3f}")

Overall Krippendorff alpha: 0.529
